# N015 · 暴力参照、穷举与性质测试

**目标：** 用独立的小规模参照检验优化算法。

**先修：** N010, N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 差分测试；小域穷举；变形性质；固定随机种子；测试不等于证明。

**配套讲解来源：** [同名逐章意见](../../comment/015_bruteforce_oracles_properties.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

你以后会写出各种"聪明的"算法（比如动态规划、滑动窗口），但你怎么知道你写的聪明算法是对的？最容易暴露 bug 的办法，是再写一个"笨但几乎不可能写错"的暴力程序，让两个程序对同一批输入各算一遍，然后逐个比对答案。只要有一次对不上，就说明其中至少有一个写错了。这一章教的就是这套流程：写暴力、写参照、把小输入全部穷举一遍、比对。

本章拿一个经典问题当练习场：**最大子数组和**。给定一个整数数组，你要找出其中一段连续的非空子数组，使这段的和尽可能大，返回这个最大的和。

举个具体到数字的例子：输入 `[-2, 3, -1]` → 输出 `3` → 为什么是 `3`？我们把所有连续非空子数组列出来：

| 候选子数组 | 和 |
|---|---|
| `[-2]` | -2 |
| `[-2,3]` | 1 |
| `[-2,3,-1]` | 0 |
| `[3]` | 3 |
| `[3,-1]` | 2 |
| `[-1]` | -1 |

六个候选里最大的是 `3`，也就是只取 `[3]` 这一段。

再看一个容易出错的边界例子：输入 `[-2, -1]` → 输出 `-1`。注意数组全是负数时，答案不是 `0`！因为题目要求子数组必须非空，你被迫选一段，只能挑亏得最少的 `[-1]`。很多错误的实现会"偷偷"允许空数组并把 0 当候选，这就是本章要抓的那类 bug。

## 2. 基础知识：先读懂这些词

- **暴力法（brute force）**：不耍任何聪明技巧，把所有可能答案一个个列出来直接算的方法。它慢，但它简单直接，所以我们把它当"标准答案"来信任。
- **参照（reference / oracle）**：一个独立的、用另一种写法实现的正确程序。因为写法不同，它和主实现不太可能犯同一个错，所以两者对得上时我们对正确性更有信心。
- **差分测试（differential testing）**：让两个独立实现吃同样的输入、比对输出的测试方法。名字吓人，意思就是"俩程序各算一遍，看答案一不一致"。
- **小域穷举（exhaustive enumeration over a small domain）**：当输入的可能情况总数很小时（比如元素只取 -1/0/1、长度不超过 4），我们可以把所有输入一个不漏地全部造出来测一遍。"小域"指的是取值范围小。
- **最大子数组问题**：上面问题描述里那个"找和最大的连续非空子数组"的问题，对应 LeetCode 53 题。
- **切片求和参照**：参照程序用 `sum(a[l:r])` 这种"整段切出来求和"的朴素写法，和主实现的"逐步累计"写法思路完全不同，正好符合"参照要换写法"的要求。
- **全负数组**：所有元素都小于 0 的数组。它是最大子数组问题最经典的边界情况，用来检验实现有没有"偷偷把空数组当候选"。
- **测试不等于证明**：测试只能说明"在这些输入上没错"，不能说明"对所有输入都没错"。哪怕一万个测试全过，也可能在第一万零一个输入上翻车。这是本章反复强调的心态。
- **变形性质**（“学习目标”部分 知识点里列出的扩展词）：不直接比对"标准答案"，而是比对"输入做了某种变形后，输出应该满足的关系"。本章的差分测试（换一种实现，输出必须一致）就是最常用的一种变形性质。
- **固定随机种子**（“学习目标”部分 知识点里列出的扩展词）：如果用随机输入做测试，就要把随机种子固定下来，让每次运行生成同一批输入。这样测试一旦报错，你才能用完全相同的输入复现问题、定位 bug。

## 3. 思路推导：从小例子开始

- **Step 1：确定候选集。** 一个子数组由起点 `left` 和终点 `right` 唯一决定（都存下标、闭区间包含两端），且必须满足 `left ≤ right`。所有满足条件的 `(left, right)` 组合就是全部候选。
- **Step 2：固定左端点，向右扩张。** 我们外层循环固定 `left`，内层让 `right` 从 `left` 一直走到数组末尾，这样每个候选子数组恰好被枚举一次。
- **Step 3：用累计和省掉重复加法。** 当 `right` 从 `l` 走到 `l+1` 时，新区间的和等于旧区间的和加上 `a[right]`，不必重新从头加一遍。这是把最朴素的 O(n³) 优化到 O(n²) 的关键一步。
- **Step 4：随时记录最大值。** 每得到一个区间的和，就与当前最优 `best` 比较，谁大留下谁。初始 `best` 取 `nums[0]`——它本身就是一个合法候选（长度为 1 的子数组），这样保证 `best` 从一开始就非空。
- **Step 5：写一个"换种写法"的参照。** 参照直接对每个切片 `a[l:r]` 调 `sum()`，写法与主实现完全不同，避免两处犯同一个错。
- **Step 6：造出全部小输入做差分测试。** 用 `product` 枚举元素取值 `{-1,0,1}`、长度 0 到 4 的所有数组（共 1+3+9+27+81=121 个），对每个非空数组比对主实现与参照的输出。

**手算演示**（对应 notebook “运行示例”部分 里 `a=[-2, 3, -1]` 的表格）：我们按"固定 left、right 右移"的顺序把六个区间走一遍。

| 轮次 | left | right（含） | 区间 | 和（累计计算过程） | best 变化 |
|---|---|---|---|---|---|
| 1 | 0 | 0 | `[-2]` | 0+(-2)=-2 | best 从初始 -2 保持 -2 |
| 2 | 0 | 1 | `[-2,3]` | -2+3=1 | best 更新为 1 |
| 3 | 0 | 2 | `[-2,3,-1]` | 1+(-1)=0 | best 保持 1 |
| 4 | 1 | 1 | `[3]` | 0+3=3（total 重新从 0 开始） | best 更新为 3 |
| 5 | 1 | 2 | `[3,-1]` | 3+(-1)=2 | best 保持 3 |
| 6 | 2 | 2 | `[-1]` | 0+(-1)=-1 | best 保持 3 |

六轮走完，`best=3`，与我们第一节手列的六个候选最大值一致。注意第 4 轮 `total` 回到 0 重新累计——这正是"换 left 就重置累计"的体现。

**再演示小数组生成器怎么工作**（取 `values=[-1,0,1]`、`max_len=1`）：n=0 时 `product(values,repeat=0)` 产出一个空的 `()`，转成 `[]`；n=1 时依次产出 `[-1]`、`[0]`、`[1]`。全部产出就是 1+3=4 个数组。把 max_len 提到 4，产出总数就是 1+3+9+27+81=121 个——测试那一节要穷举的就是这 121 个。

## 4. 核心代码：max_subarray_bruteforce

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def max_subarray_bruteforce(nums):
    if not nums:
        raise ValueError('requires a nonempty array')
    best = nums[0]
    for left in range(len(nums)):
        total = 0
        for right in range(left, len(nums)):
            total += nums[right]
            best = max(best, total)
    return best
```

### 逐段读懂代码

“分段实现”部分 一共两个函数，我们逐个看。

### 函数 1：`max_subarray_bruteforce`（主暴力实现）

先看前半段（入口检查与初始化）：

```python
from itertools import product

def max_subarray_bruteforce(nums):
    if not nums:
        raise ValueError('requires a nonempty array')
    best = nums[0]
```

- `from itertools import product`：引入笛卡尔积工具，下面第二个函数用它生成所有组合，这里先记住"product 是造所有排列组合的"。
- `if not nums: raise ValueError(...)`：挡住空数组。题目规定子数组必须非空，空输入没有合法答案，与其返回一个误导性的数字，不如直接报错，把问题暴露在明面上。
- `best=nums[0]`：把最优答案初始化为第一个元素。这样写的好处是 `best` 一开始就是一个真实存在的候选（子数组 `[nums[0]]` 的和），而不是 `0`——如果初始化成 0，全负数组就会错误地返回 0。

再看主循环（两层枚举 + 逐步累计）：

```python
for left in range(len(nums)):
        total=0
        for right in range(left,len(nums)):
            total+=nums[right]
            best=max(best,total)
    return best
```

- 外层 `for left in range(len(nums))`：枚举每个可能的起点。
- `total=0`：每换一个起点，累计和清零，从头累计。这一行在 left 循环体内、right 循环体外，位置很关键：清零太早会丢掉累计，清零太晚会带进上一个起点的结果。
- 内层 `for right in range(left,len(nums))`：终点从起点一路走到数组末尾，保证 `right ≥ left`。
- `total+=nums[right]`：新区间的和 = 旧区间的和 + 新进来这一个元素。这就是 Step 3 说的"复用前一段的和"，省掉了内层再套一层求和循环。
- `best=max(best,total)`：谁大留谁。等价于 `if total>best: best=total` 的紧凑写法。
- `return best`：所有候选都比过了，返回冠军。

### 函数 2：`enumerate_small_arrays`（小输入生成器）

```python
def enumerate_small_arrays(values, max_len):
    for n in range(max_len + 1):
        for row in product(values, repeat=n):
            yield list(row)
```

- `for n in range(max_len+1)`：枚举长度 0,1,2,...,max_len。注意 `range(max_len+1)` 里的 `+1`，否则会漏掉最长的那种。
- `for row in product(values,repeat=n)`：从 `values` 里可重复地取 n 个数排成一排，穷举所有长度为 n 的组合。例如 `values=[-1,1]`、`n=2` 时依次产出 `(-1,-1),(-1,1),(1,-1),(1,1)`。
- `yield list(row)`：这是生成器写法，产出一个就交出一个，不把所有组合一次性堆在内存里。注意 `product` 产出的是元组，这里转成 list，方便后续当数组用。顺带一提：`n=0` 时会产出一个空数组 `[]`，这是故意保留的——测试代码里会用 `if a:` 把它跳过。

## 5. 分段实现：按顺序运行

**本章接口：** `max_subarray_bruteforce(nums)`、`enumerate_small_arrays(values, max_len)`

### 导入本章使用的库

In [1]:
from itertools import product

### max_subarray_bruteforce

In [2]:
def max_subarray_bruteforce(nums):
    if not nums:
        raise ValueError('requires a nonempty array')
    best = nums[0]
    for left in range(len(nums)):
        total = 0
        for right in range(left, len(nums)):
            total += nums[right]
            best = max(best, total)
    return best

### enumerate_small_arrays

In [3]:
def enumerate_small_arrays(values, max_len):
    for n in range(max_len + 1):
        for row in product(values, repeat=n):
            yield list(row)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[-2,3,-1]
show_table(['left','right（含）','区间','和'],[(l,r,a[l:r+1],sum(a[l:r+1])) for l in range(len(a)) for r in range(l,len(a))])

left,right（含）,区间,和
0,0,[-2],-2
0,1,"[-2, 3]",1
0,2,"[-2, 3, -1]",0
1,1,[3],3
1,2,"[3, -1]",2
2,2,[-1],-1


## 7. 正确性、适用条件与复杂度

每个非空连续区间唯一对应一组 left≤right，枚举覆盖所有候选；累计和只复用同一起点的前一段和，不改变候选值。

**代价：** 主暴力 O(n²) 时间、O(1) 辅助空间；切片参照 O(n³)，只用于小规模验证。

### 展开理解

**为什么是对的：** 一个子数组说到底就是"从 left 开始、到 right 结束的一段"，只要 left 和 right 都在数组范围内且 left 不超过 right，它就是一个合法候选。我们的两层循环恰好把每一对满足条件的 `(left, right)` 都走了一遍，一个不多也一个不少，所以最优解必然出现在被比较过的候选里。另外，累计和没有偷工减料：`total` 始终精确等于当前区间 `nums[left..right]` 的真实和，因为它只是在正确的前一段和上加了正确的新元素。最后，"非空"这个约束由两层循环的结构天然保证（right 从 left 出发，区间至少有一个元素），再加上函数开头就把空输入挡掉了，所以全负数组也只会返回某个真实元素的组合，绝不会返回空数组才有的 0。

**复杂度是多少：** 主暴力有两层循环，外层 left 走 n 步，内层 right 平均也走约 n/2 步，每步只做加法和比较这种常数时间的活，所以总时间是 O(n²)，额外只用了 `best` 和 `total` 两个变量，空间是 O(1)。切片参照对每个区间都要从头到尾重新加一遍，是 O(n³)。举个具体的数：n=1000 时，主暴力大概要做 100 万次基本操作，一眨眼就算完；但 n=10⁵ 时 O(n²) 就要约 100 亿次操作，会明显卡住——这正是暴力法只配当"小输入检验员"、真正解题还需要优化算法的原因。

**两种典型错法，说说测试是怎么抓住它们的：**

- 错法一：把 `best` 初始化成 `0`。这样的实现对 `[-2,-1]` 会返回 0（因为任何负数候选都比不过初始的 0），而正确答案是 -1。第六节的第 1 条 assert 直接抓住它。
- 错法二：把参照里 r 的范围写成 `range(l,len(a))`（漏了 +1）。这会把"取到最后一个元素"的那些切片全部漏掉，参照自己就错了。差分穷举一跑，主实现与参照在大量数组上对不上，同样会炸——这也是为什么参照要"简单到不至于写错"，以及为什么对不上时两边都要查。

## 8. 单元测试：每个用例在检查什么

```python
assert max_subarray_bruteforce([-2, -1]) == -1

assert max_subarray_bruteforce([0]) == 0

for a in enumerate_small_arrays([-1, 0, 1], 4):
    if a:
        ref = max((sum(a[l:r]) for l in range(len(a)) for r in range(l + 1, len(a) + 1)))
        assert max_subarray_bruteforce(a) == ref

print('N015: 所有本章断言通过')
```

- 第 1 条 `assert max_subarray_bruteforce([-2,-1])==-1`：这是**特殊值/边界测试**。输入是全负数组，正确答案必须是 -1（选亏得最少的 `[-1]`）。如果实现偷偷允许空数组，就会错误地返回 0，这条 assert 当场抓住它。
- 第 2 条 `assert max_subarray_bruteforce([0])==0`：这是**最小规模测试**。数组只有 1 个元素，没有选择的余地，答案只能就是这个元素本身。它能暴露"循环边界写错导致连单元素数组都处理不好"的低级错误。
- 第 3 段 for 循环是本章的重头戏——**差分穷举测试**。`enumerate_small_arrays([-1,0,1],4)` 生成元素取值 -1/0/1、长度 0~4 的全部 121 个数组。对每个数组做三件事：
  - `if a:` 先跳过空数组——空数组会被主实现的入口检查拒绝（报错），这是预期行为，不属于"答案对不对"的比拼范围。
  - `ref=...` 用"切片求和"的另一种写法算参照答案。注意它的区间写法是 `a[l:r]`（左闭右开），所以 r 的范围是 `l+1` 到 `len(a)+1`，和主实现的闭区间下标一一对应，两边的候选集其实完全相同。
  - `assert max_subarray_bruteforce(a)==ref` 要求主实现与参照在每一个数组上都给出相同结果。121 个数组任何一个对不上，测试当场炸出来。

`print('N015: 所有本章断言通过')` 只是全过之后打印一句确认，让你肉眼看到"这批都过了"。

In [5]:
assert max_subarray_bruteforce([-2, -1]) == -1

assert max_subarray_bruteforce([0]) == 0

for a in enumerate_small_arrays([-1, 0, 1], 4):
    if a:
        ref = max((sum(a[l:r]) for l in range(len(a)) for r in range(l + 1, len(a) + 1)))
        assert max_subarray_bruteforce(a) == ref

print('N015: 所有本章断言通过')

N015: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 枚举{-1,0,1}长度不超过5的非空数组。

**练习 2：** 加入全负反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（枚举 {-1,0,1} 长度不超过 5 的非空数组）**：思路方向——这题就是把本章的差分测试流程照搬一遍、只是把 `max_len` 从 4 提到 5（数组总数变成 1+3+9+27+81+243=364 个）。提示：先用一个小数组（比如 `[-1,1]`）手算出最大子数组和你期望的答案，写下来，再跑程序对照；你手算的那一步就是"验证参照"的雏形。别忘了明确写出"输入是什么、输出是什么、空数组怎么处理"这三件事。
- **练习 2（加入全负反例）**：思路方向——专门构造几个所有元素都是负数的数组（如 `[-3,-1,-2]`），先手算：全负时最优解是"绝对值最小的那个元素单独成段"，本例答案是 -1。提示：让差分测试的取值池里出现"不含 0 和正数"的负数组合更省事的办法，是直接把 `values` 换成两个不同的负数（比如 `[-2,-1]`）再穷举一轮；重点观察你的实现会不会返回 0——一旦返回 0，说明它把空数组当候选了，回头检查 `best` 的初始化。

两道练习验收时的共同要求：把"输入、输出、边界"三件事写明白，并准备口头解释"为什么全负数组的答案一定不是 0"——能讲出这句，才算真懂了"非空"这个约束。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from itertools import product

def max_subarray_bruteforce(nums):
    if not nums:
        raise ValueError('requires a nonempty array')
    best = nums[0]
    for left in range(len(nums)):
        total = 0
        for right in range(left, len(nums)):
            total += nums[right]
            best = max(best, total)
    return best

def enumerate_small_arrays(values, max_len):
    for n in range(max_len + 1):
        for row in product(values, repeat=n):
            yield list(row)

# 示例与回归测试
assert max_subarray_bruteforce([-2, -1]) == -1

assert max_subarray_bruteforce([0]) == 0

for a in enumerate_small_arrays([-1, 0, 1], 4):
    if a:
        ref = max((sum(a[l:r]) for l in range(len(a)) for r in range(l + 1, len(a) + 1)))
        assert max_subarray_bruteforce(a) == ref

print('N015: 所有本章断言通过')

N015: 所有本章断言通过


## 11. 代表题与迁移

- **53. Maximum Subarray（最大子数组）**：这题练的就是本章的示例问题本身——找和最大的连续非空子数组。本章给了它一个 O(n²) 的暴力基线，学习后面章节后你可以拿 Kadane 算法或分治法回来对比，并用本章的差分测试方法验证你的优化实现。
- 迁移建议：以后任何一章你写了新的优化解法，都可以回来借用本章的思路——为它写一个笨参照，再用 `enumerate_small_arrays` 式的小域穷举做一轮差分比对，这是成本最低的正确性保险。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。